# Reinforcement learning with verifiable rewards

Supervised fine-tuning shows a model the answer and asks it to copy. For a lot of tasks we cannot write the answer down, but we can check one: a program either passes its tests or it does not, and a maths answer either matches or it does not. Reinforcement learning with verifiable rewards (RLVR) uses that. The model writes several answers, a checker scores them, and training makes the better ones more likely.

The algorithm most RLVR runs use today is GRPO, from DeepSeekMath ([Shao et al., 2024](https://arxiv.org/abs/2402.03300)). It looks like a handful of tricks stacked on top of each other, but each one has a reason, and all of them are small changes to the oldest policy-gradient estimator, REINFORCE ([Williams, 1992](https://link.springer.com/article/10.1007/BF00992696)). In this notebook we derive GRPO's gradient one step at a time. First we use a problem small enough to compute everything exactly. Then we work out the gradient of Dew's `GRPOObjective` by hand on a toy policy and check it against the one Dew computes. Finally we train that policy and watch its reward go up.

Everything here runs on a CPU in a few minutes.

**Notation.** A prompt is $q$. The policy $\pi_\theta$ samples a group of $G$ completions $y^{(1)}, \dots, y^{(G)}$ for it, and $y^{(i)}_t$ is token $t$ of completion $i$. A checker gives completion $i$ a reward $R^{(i)}$. The mask $m^{(i)}_t$ is 1 on the tokens the policy sampled and 0 on the prompt. $A$ is an advantage, $\rho$ an importance ratio, $\varepsilon$ the clip range and $\beta$ the weight of the KL penalty.

## Install dependencies

Dew from GitHub. Nothing here needs a GPU.

In [ ]:
%pip install -q "dewml @ git+https://github.com/AshishKumar4/dew"

## Settings

The toy task in the second half uses digits, so the vocabulary is 10 tokens. Each prompt is 4 digits and the policy writes 6 more.

In [ ]:
import os

VOCAB = 10
PROMPT_LENGTH = 4
NEW_TOKENS = 6
PROMPTS = 8          # prompts per batch
GROUPS = 8           # completions per prompt, G
STEPS = 150          # rollouts in the training run
UPDATES = 2          # gradient steps on each rollout
LEARNING_RATE = 0.05
BETA = 0.02          # KL weight
SEED = 0

# The tutorial test sets DEW_TUTORIAL_SMOKE=1 to run every cell in minutes on a CPU.
if os.environ.get("DEW_TUTORIAL_SMOKE") == "1":
    PROMPTS, GROUPS, STEPS = 4, 4, 5

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

print(jax.devices())

## One decision: REINFORCE

Start with the smallest case there is. The policy picks one of $K$ answers, $a$, with probability $\pi_\theta(a) = \mathrm{softmax}(\theta)_a$, and a checker gives it a reward $R(a)$. We want to raise the expected reward

$$J(\theta) = \mathbb{E}_{a \sim \pi_\theta}[R(a)] = \sum_a \pi_\theta(a)\, R(a).$$

Its gradient is $\sum_a R(a)\, \nabla \pi_\theta(a)$. Since $\nabla \pi = \pi\, \nabla \log \pi$, we can write that as an expectation:

$$\nabla J = \sum_a \pi_\theta(a)\, R(a)\, \nabla \log \pi_\theta(a) = \mathbb{E}_{a \sim \pi_\theta}\big[R(a)\, \nabla \log \pi_\theta(a)\big].$$

That last form is the whole point. We can estimate it by sampling answers and averaging $R(a)\, \nabla \log \pi_\theta(a)$, and we only ever need the reward of answers we actually tried. This is REINFORCE. For a softmax, $\nabla_\theta \log \pi_\theta(a) = \mathrm{onehot}(a) - \pi_\theta$.

Below there are five answers, and only answer 3 is right. We compare the exact gradient, from `jax.grad` of $J$, with REINFORCE averaged over many samples.

In [ ]:
K, RIGHT = 5, 3
theta = jax.random.normal(jax.random.key(SEED), (K,)) * 0.5
reward = (jnp.arange(K) == RIGHT).astype(jnp.float32)

def expected_reward(theta):
    return jnp.sum(jax.nn.softmax(theta) * reward)

exact = jax.grad(expected_reward)(theta)
probs = jax.nn.softmax(theta)

def reinforce(key, samples):
    answers = jax.random.categorical(key, theta, shape=(samples,))
    score = jax.nn.one_hot(answers, K) - probs           # grad of log pi(a), one row per sample
    return jnp.mean(reward[answers][:, None] * score, axis=0)

np.set_printoptions(precision=4, suppress=True)
print("pi(a)            ", np.asarray(probs))
print("exact gradient   ", np.asarray(exact))
print("REINFORCE, 100k  ", np.asarray(reinforce(jax.random.key(1), 100_000)))
print("REINFORCE, 8     ", np.asarray(reinforce(jax.random.key(2), 8)))

With 100,000 samples the estimate matches. With 8 it points somewhere in the right direction, but it is noisy, and 8 is closer to what we can afford when every sample is a whole generated answer.

## Baselines and group advantages

We can subtract any baseline $b$ that does not depend on the sampled answer without changing the expected gradient, because

$$\mathbb{E}_{a \sim \pi_\theta}[\nabla \log \pi_\theta(a)] = \sum_a \nabla \pi_\theta(a) = \nabla \sum_a \pi_\theta(a) = \nabla 1 = 0.$$

So we can weight each sample by $R(a) - b$ instead of $R(a)$. A good $b$ is the reward we expected anyway. Then a right answer that was already likely gets a small push, and a wrong answer gets pushed down rather than just ignored. GRPO gets $b$ without a learned value function: it samples a group of $G$ answers to the same prompt and uses the group's mean reward. It also divides by the group's standard deviation, so every prompt's advantages have about the same scale, whether its rewards are spread out or nearly equal:

$$A^{(i)} = \frac{R^{(i)} - \operatorname{mean}_j R^{(j)}}{\operatorname{std}_j R^{(j)} + \epsilon}.$$

(Using the group's own mean makes the baseline depend on the sample a little, so the estimate picks up a small bias of order $1/G$. Dew's `rloo_advantage`, which leaves each sample out of its own baseline, avoids that.)

We draw 4,000 groups of 8 and measure how far each estimator lands from the exact gradient. The normalized one is on a different scale, so for it we measure the cosine between estimate and exact gradient.

In [ ]:
def group_estimates(key, groups, size, baseline, normalize):
    answers = jax.random.categorical(key, theta, shape=(groups, size))
    r = reward[answers]
    advantage = r - (r.mean(axis=1, keepdims=True) if baseline else 0.0)
    if normalize:
        advantage = advantage / (r.std(axis=1, ddof=1, keepdims=True) + 1e-6)
    score = jax.nn.one_hot(answers, K) - probs
    return jnp.mean(advantage[..., None] * score, axis=1)    # one estimate per group

def cosine(estimates):
    return jnp.mean(estimates @ exact / (jnp.linalg.norm(estimates, axis=1) * jnp.linalg.norm(exact) + 1e-12))

key = jax.random.key(3)
for name, baseline, normalize in (("R", False, False), ("R - mean", True, False), ("GRPO's A", True, True)):
    estimates = group_estimates(key, 4000, 8, baseline, normalize)
    error = float(jnp.mean(jnp.sum((estimates - exact) ** 2, axis=1)))
    print(f"{name:10s} squared error {error:.5f}   cosine with the exact gradient {float(cosine(estimates)):.3f}")

Subtracting the mean cuts the error, and the direction gets closer to the true gradient. Normalizing changes the scale (so the error row isn't comparable for it), but not the direction.

A group where every answer got the same reward has every $A^{(i)} = 0$ and teaches nothing. That is why RLVR wants tasks the model gets right sometimes, but not always.

## From one answer to a sequence

A completion is a sequence of tokens, and its probability is a product, so

$$\log \pi_\theta(y \mid q) = \sum_t \log \pi_\theta(y_t \mid q, y_{<t}).$$

The gradient of a whole completion is the sum of its tokens' gradients, and each sampled token gets its completion's advantage. The prompt's tokens were not sampled, so they get no gradient; that's what the mask $m$ is for. Dew, like verl, averages over all the sampled tokens of the batch (the "token mean"). Written as a loss to minimise:

$$L(\theta) = -\frac{1}{\sum_{i,t} m^{(i)}_t} \sum_{i,t} m^{(i)}_t\, A^{(i)} \log \pi_\theta(y^{(i)}_t \mid q, y^{(i)}_{<t}).$$

Let $z$ be the logits at the position that predicts $y_t$. Since $\partial \log \mathrm{softmax}(z)_y / \partial z = \mathrm{onehot}(y) - \mathrm{softmax}(z)$, the gradient for that one token is

$$\frac{\partial L}{\partial z} = -\frac{m_t}{\sum m}\, A\, \big(\mathrm{onehot}(y_t) - \mathrm{softmax}(z)\big).$$

## Reusing samples: the ratio, the clip and the KL

Generating is the expensive part of RLVR, so GRPO takes more than one gradient step on each batch of samples. After the first step the samples no longer come from the current policy $\pi_\theta$; they come from the policy that drew them, $\pi_{\theta_\text{old}}$. Importance sampling corrects for that by weighting each token by

$$\rho_t = \frac{\pi_\theta(y_t \mid q, y_{<t})}{\pi_{\theta_\text{old}}(y_t \mid q, y_{<t})},$$

since $\mathbb{E}_{\pi_\text{old}}[\rho\, A\, \nabla \log \pi_\theta] = \mathbb{E}_{\pi_\theta}[A\, \nabla \log \pi_\theta]$. The per-token loss becomes $-\rho_t A$, and its gradient with respect to $\log \pi_\theta(y_t)$ is $-\rho_t A$. At $\theta = \theta_\text{old}$ the ratio is 1 and we are back to REINFORCE with advantages.

Taking several steps on the same samples can push the policy far from the one that drew them, and then the ratio and the estimate both get unreliable. PPO's clip ([Schulman et al., 2017](https://arxiv.org/abs/1707.06347)) puts a stop to that. Each token's term is

$$\max\big(-\rho_t A,\; -\operatorname{clip}(\rho_t, 1 - \varepsilon_\text{low}, 1 + \varepsilon_\text{high})\, A\big).$$

When the clipped branch is the larger one, the term no longer depends on $\theta$, so that token's gradient is zero. In words: once a token's probability has moved far enough in the direction its advantage asks for, it stops pushing. Moves in the other direction are never clipped. Dew also caps a negative-advantage term at $-cA$ with $c = 3$ (the dual clip of [Ye et al., 2020](https://arxiv.org/abs/1912.09729)), so a single token whose ratio has blown up cannot take over the step.

Last, GRPO adds a penalty for drifting from a frozen reference policy $\pi_\text{ref}$, usually the model you started from, with weight $\beta$. Per token it uses the k3 estimator of the KL divergence,

$$k_3 = \frac{\pi_\text{ref}}{\pi_\theta} - \log \frac{\pi_\text{ref}}{\pi_\theta} - 1 \;\ge 0, \qquad \frac{\partial k_3}{\partial \log \pi_\theta} = 1 - \frac{\pi_\text{ref}}{\pi_\theta}.$$

Put together, the gradient of the loss with respect to token $t$'s logits is

$$\frac{\partial L}{\partial z_t} = \frac{m_t}{\sum m}\Big(g_t + \beta\big(1 - \tfrac{\pi_\text{ref}(y_t)}{\pi_\theta(y_t)}\big)\Big)\big(\mathrm{onehot}(y_t) - \mathrm{softmax}(z_t)\big),$$

where $g_t = -\rho_t A$ if the token's term is unclipped, and $g_t = 0$ if a clip holds it.

That is all of GRPO's gradient. Next we check it against Dew's.

## A task with a checker

The policy reads 4 random digits and writes 6 more. Its job is to keep counting up from the last digit, modulo 10: after `7 2 9 4` the right continuation is `5 6 7 8 9 0`. The checker gives the fraction of written digits that are one more than the digit before them. It's a verifiable reward in miniature: easy to check, and the model has to find out what is being asked of it.

The policy is as small as it can be while still being a language model. Its next-token logits depend only on the current token, through a 10 by 10 table $W$, so the logits at a position are row $x_t$ of $W$. Because of that, the gradient with respect to $W$ is just the per-token logit gradients above, each added into the row of the token that was read. The model gives Dew its states (`hidden_states`, here a one-hot of the token) and its head as one matrix (`output_table`), which is how Dew scores a decoder's tokens. `init_cache` is there because generation threads a cache through every model; this one keeps nothing in it.

In [ ]:
from flax import linen as nn

from dew.nn.protocols import OutputTable


class Bigram(nn.Module):
    vocab_size: int

    def setup(self):
        self.table = self.param("table", nn.initializers.normal(0.1), (self.vocab_size, self.vocab_size))

    def hidden_states(self, tokens, train=False, **fields):
        return jax.nn.one_hot(tokens, self.vocab_size)

    def output_table(self):
        return OutputTable(self.table, vocab_major=False)

    @nn.compact
    def init_cache(self, batch_size):
        self.variable("cache", "index", lambda: jnp.zeros((batch_size,), jnp.int32))

    def __call__(self, tokens, train=False, decode=False, attention_mask=None, **fields):
        return self.hidden_states(tokens) @ self.table

A prompt batch carries the prompt ids and three strings for the reward: where the row came from, its ground truth (here the last prompt digit, which the first written digit must follow), and any extra information. They are stored as fixed-width rows of UTF-8 bytes so they can travel in an array batch.

In [ ]:
from dew.data.prompts import INFO_KEY, LENGTH_KEY, PROMPT_KEY, SOURCE_KEY, TRUTH_KEY


def text_rows(texts, width=8):
    rows = np.zeros((len(texts), width), np.int32)
    for row, text in enumerate(texts):
        encoded = np.frombuffer(text.encode(), np.uint8)
        rows[row, :len(encoded)] = encoded
    return rows


def prompt_batch(rng):
    prompts = rng.integers(0, VOCAB, (PROMPTS, PROMPT_LENGTH)).astype(np.int32)
    return {PROMPT_KEY: prompts, LENGTH_KEY: np.full(PROMPTS, PROMPT_LENGTH, np.int32),
            SOURCE_KEY: text_rows(["count"] * PROMPTS),
            TRUTH_KEY: text_rows([str(prompt[-1]) for prompt in prompts]),
            INFO_KEY: text_rows([""] * PROMPTS)}


def counts_up(source, completion, truth, info):
    previous, right = int(truth), 0
    for digit in map(int, completion.split()):
        right += digit == (previous + 1) % VOCAB
        previous = digit
    return right / NEW_TOKENS


print(counts_up("count", "5 6 7 8 9 0", "4", ""), counts_up("count", "5 6 1 2 9 0", "4", ""))

`GRPOObjective` is Dew's GRPO loss, and `SampledRollout` draws `GROUPS` completions for each prompt, scores them with the checker, computes the group advantages and packs everything into the batch the loss reads. `seq_len` is one less than a row's width, prompt plus completion, as for any language-model objective. `beta=BETA` makes the objective keep a frozen copy of the starting weights as $\pi_\text{ref}$.

In [ ]:
from types import SimpleNamespace

from dew.objectives.base import Step
from dew.objectives.rl import GRPOObjective, SampledRollout
from dew.sampling import Sampling

objective = GRPOObjective(Bigram(VOCAB), seq_len=PROMPT_LENGTH + NEW_TOKENS - 1, beta=BETA)
start = objective.init(jax.random.key(SEED))
rollout = SampledRollout(objective, counts_up, groups=GROUPS, max_new_tokens=NEW_TOKENS,
                         sampling=Sampling(temperature=1.0))
rng = np.random.default_rng(SEED)
batch = rollout(SimpleNamespace(variables=start, updates=0), prompt_batch(rng), jax.random.key(1))

print({name: value.shape for name, value in batch.items()})
print("first row:      ", batch["input_ids"][0])
print("response mask:  ", batch["response_mask"][0])
print("advantage:      ", batch["advantages"][0].round(3))
print("mean reward:    ", rollout.metrics["reward/mean"])

Each row is one completion: its prompt, then the digits the policy wrote. The response mask is 1 on the written digits, and every one of them carries its completion's advantage. `old_log_probs` holds $\log \pi_{\theta_\text{old}}$ of each written digit, recorded while sampling.

First, the advantages. We recompute every reward with the checker, group them by prompt and normalize them ourselves.

In [ ]:
rows = batch["input_ids"]
written = batch["response_mask"] != 0
rewards = np.array([counts_up("", " ".join(map(str, row[mask])), str(row[PROMPT_LENGTH - 1]), "")
                    for row, mask in zip(rows, written)])
grouped = rewards.reshape(-1, GROUPS)
ours = ((grouped - grouped.mean(axis=1, keepdims=True))
        / (grouped.std(axis=1, ddof=1, keepdims=True) + 1e-6)).reshape(-1)
np.testing.assert_allclose(batch["advantages"][:, -1], ours, rtol=1e-5, atol=1e-6)
print("rewards of the first prompt's group:", grouped[0])
print("the advantages match")

Now the gradient. To make the ratio and the clip do something, we first take one gradient step on this batch, which is what GRPO's second update on a batch sees: $\theta \ne \theta_\text{old}$. Then Dew computes the gradient of its loss at the new weights with `jax.grad`, and we compute it by hand from the formula above, one written token at a time.

In [ ]:
import optax

step = Step(step=jnp.zeros((), jnp.int32), key=jax.random.key(0), ema=start)


def loss(params, batch):
    statistics, _ = objective.loss(params, batch, step)
    return statistics.mean()[0]


optimizer = optax.adam(0.3)
grads = jax.grad(loss)(start, batch)
update, _ = optimizer.update(grads, optimizer.init(start), start)
moved = optax.apply_updates(start, update)

dews = np.asarray(jax.grad(loss)(moved, batch)["params"]["table"])


def by_hand(params, reference, batch, low=0.2, high=0.2, dual=3.0):
    table = np.asarray(params["params"]["table"], np.float64)
    frozen = np.asarray(reference["params"]["table"], np.float64)
    ids, mask = batch["input_ids"], batch["response_mask"]
    gradient, total, clipped = np.zeros_like(table), mask.sum(), 0
    for row, column in zip(*np.nonzero(mask)):
        read, written = ids[row, column - 1], ids[row, column]
        probs = np.exp(table[read] - np.logaddexp.reduce(table[read]))
        reference_probs = np.exp(frozen[read] - np.logaddexp.reduce(frozen[read]))
        advantage = batch["advantages"][row, column]
        ratio = probs[written] / np.exp(batch["old_log_probs"][row, column])
        unclipped, held = -ratio * advantage, -np.clip(ratio, 1 - low, 1 + high) * advantage
        term = max(unclipped, held)
        active = unclipped >= held and not (advantage < 0 and term > -dual * advantage)
        clipped += not active
        g = (-ratio * advantage if active else 0.0) + BETA * (1 - reference_probs[written] / probs[written])
        gradient[read] += g / total * (np.eye(VOCAB)[written] - probs)
    return gradient, clipped


ours, clipped = by_hand(moved, start, batch)
print(f"{clipped} of {int(batch['response_mask'].sum())} written tokens are held by a clip")
print("largest difference:", np.abs(dews - ours).max(), " largest entry:", np.abs(ours).max())
np.testing.assert_allclose(dews, ours, rtol=1e-4, atol=1e-7)

The two agree to float32 rounding, clipped tokens included. Dew's loss is exactly the derivation: the group advantage, the ratio to the sampling policy, the clips and the KL to the reference.

## Training

Now we train. Each step draws a fresh batch of prompts, samples `GROUPS` completions for each, and takes `UPDATES` Adam steps on that rollout. The first sees ratios of 1; the second sees the policy it just moved, which is where the ratio and the clip earn their keep.

In [ ]:
params = start
optimizer = optax.adam(LEARNING_RATE)
state = optimizer.init(params)
gradient = jax.jit(jax.grad(loss))
history = []
for index in range(STEPS):
    batch = rollout(SimpleNamespace(variables=params, updates=index), prompt_batch(rng),
                    jax.random.fold_in(jax.random.key(2), index))
    history.append(rollout.metrics["reward/mean"])
    for _ in range(UPDATES):
        update, state = optimizer.update(gradient(params, batch), state, params)
        params = optax.apply_updates(params, update)

plt.figure(figsize=(6, 3))
plt.plot(history)
plt.xlabel("rollout")
plt.ylabel("mean reward")
plt.ylim(0, 1)
plt.show()
print("mean reward of the first 10 rollouts:", np.mean(history[:10]).round(3),
      " of the last 10:", np.mean(history[-10:]).round(3))

A random policy gets about a tenth of the digits right, and the trained one counts. Its table says the same thing: in each row, the largest logit sits one column to the right of the row.

In [ ]:
from dew.sampling import generate

table = np.asarray(params["params"]["table"])
print("each digit's most likely next digit:", table.argmax(axis=1))
prompt = jnp.asarray([[7, 2, 9, 4]], jnp.int32)
greedy = generate(objective.model, params, prompt, NEW_TOKENS, key=0, sampling=Sampling(temperature=0))
print("7 2 9 4 ->", greedy.tokens[0, PROMPT_LENGTH:])

## What changes for a real model

Nothing in the loss. A transformer policy gets its logits from a deeper network, but `GRPOObjective` computes the same per-token terms and the same gradient. What changes is everything around the loss. The checker runs programs in a sandbox (`dew.rl.sandbox`). Rollouts come from a serving engine, Dew's own `Server` or vLLM or SGLang, while the trainer updates, so a batch is often one update stale; the ratio above corrects for that. And long or multi-turn episodes are packed into rows with masks. `examples/train_rlvr.py` runs all of this, training a small instruct model to write Python programs that pass their tests.